# SignSense AI — Combined Model Training Notebook

Train **all three models** (MLP + BiLSTM + MobileNetV3 CNN) in a single Colab session.

| Model | Input | Target Accuracy | Est. Time |
|-------|-------|-----------------|------------|
| MLP | 63-dim landmark vector | > 95% | ~15 min |
| BiLSTM | 30×63 sequence | > 85% | ~45 min |
| MobileNetV3 CNN | 224×224 hand crop | > 90% | ~2 h |

**Total estimated time: ~3 h on T4 GPU**

---
### Before you start
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells top to bottom — do NOT skip any cell
3. If Colab disconnects: re-run from Cell 1. All training is resume-safe via Drive checkpoints.

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_MODELS_DIR = '/content/drive/MyDrive/SignSense/models'
DRIVE_DATA_DIR   = '/content/drive/MyDrive/SignSense/data'
DRIVE_CROPS_DIR  = '/content/drive/MyDrive/SignSense/crops'
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
os.makedirs(DRIVE_DATA_DIR,   exist_ok=True)
os.makedirs(DRIVE_CROPS_DIR,  exist_ok=True)
print(f'✅ Drive mounted.')
print(f'   Models → {DRIVE_MODELS_DIR}')
print(f'   Data   → {DRIVE_DATA_DIR}')
print(f'   Crops  → {DRIVE_CROPS_DIR}')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# mediapipe ≥0.10.18 requires protobuf ≥5.28 (Colab's TF 2.20 is compatible)
!pip install -q 'protobuf>=5.28.0' 'mediapipe>=0.10.18' scikit-learn tqdm albumentations

import importlib, sys
if 'google.protobuf' in sys.modules:
    print('⚠️  protobuf is already loaded in memory.')
    print('   Go to Runtime → Restart session, then re-run ALL cells from Cell 1.')
else:
    print('✅ Dependencies installed — continue to Cell 3.')

In [ ]:
# ── Cell 3: Upload backend code ───────────────────────────────────────────────
# Run  .\notebooks\create_colab_zip.ps1  locally first to create backend_colab.zip
# Then upload it here (file picker will appear).

from google.colab import files
import os, sys, zipfile, urllib.request, shutil

BACKEND_PATH = '/content/backend'
GITHUB_RAW   = 'https://raw.githubusercontent.com/prateek1756/sign-language-detection/master'

if not os.path.exists(BACKEND_PATH):
    print('Upload backend_colab.zip when the file picker appears...')
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content')
    print(f'Extracted {zip_name} to /content/')
else:
    print('backend/ already exists, skipping upload.')

# Always pull latest source files from GitHub so Colab runs fixed versions
SRC_FILES = [
    'backend/src/preprocess.py',
    'backend/src/model.py',
    'backend/src/train.py',
    'backend/src/evaluate.py',
    'backend/configs/training_config.py',
]
print('\nPulling latest source files from GitHub...')
for rel_path in SRC_FILES:
    url  = f'{GITHUB_RAW}/{rel_path}'
    dest = f'/content/{rel_path}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(url, dest)
    print(f'  ✅ {rel_path}')

# ── Nuke __pycache__ so stale bytecode never runs ────────────────────────────
for cache_dir in [f'{BACKEND_PATH}/src/__pycache__',
                  f'{BACKEND_PATH}/configs/__pycache__']:
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)
        print(f'  🗑  Cleared {cache_dir}')

# ── Verify the Keras 3 fix is present ───────────────────────────────────────
with open(f'{BACKEND_PATH}/src/model.py') as _f:
    _src = _f.read()
assert 'set_global_policy' not in _src, (
    '❌ model.py still has set_global_policy — GitHub pull may have failed. '
    'Try re-running this cell.'
)
print('  ✅ model.py verified — Keras 3 fix is present')

sys.path.insert(0, BACKEND_PATH)

# Verify import
from configs.training_config import ASL_CLASSES, NUM_CLASSES
print(f'\n✅ Import OK — {NUM_CLASSES} classes: {ASL_CLASSES[:5]}...')

In [ ]:
# ── Cell 4: Download & preprocess ASL dataset ─────────────────────────────────
# Prerequisites:
#   - Cell 3 must have run (backend/ is at /content/backend)
#   - Upload kaggle.json via Files panel BEFORE running this cell
#     Get it from: https://www.kaggle.com/settings → API → Create New Token
#
# On reconnect: if Drive cache exists this cell skips download automatically.

import os, sys, shutil, urllib.request
import numpy as np
from pathlib import Path

BACKEND_PATH  = '/content/backend'
RAW_ASL_DIR   = f'{BACKEND_PATH}/data/raw/ASL'
PROCESSED_DIR = f'{BACKEND_PATH}/data/processed/ASL'
PROCESSED_NPY = f'{PROCESSED_DIR}/landmarks_all.npy'
LABELS_NPY    = f'{PROCESSED_DIR}/labels_all.npy'
CROPS_DIR     = f'{PROCESSED_DIR}/crops'
KAGGLE_JSON   = '/content/kaggle.json'
KAGGLE_DEST   = '/root/.kaggle/kaggle.json'
DOWNLOAD_DIR  = '/content/asl_data'

# ── Restore from Drive cache if available ────────────────────────────────────
restored = False
if not os.path.exists(PROCESSED_NPY):
    drive_x = f'{DRIVE_DATA_DIR}/landmarks_all.npy'
    if os.path.exists(drive_x):
        print('Restoring preprocessed landmark data from Drive cache...')
        os.makedirs(PROCESSED_DIR, exist_ok=True)
        for fname in ['landmarks_all.npy', 'labels_all.npy', 'class_map.json']:
            src = f'{DRIVE_DATA_DIR}/{fname}'
            dst = f'{PROCESSED_DIR}/{fname}'
            if os.path.exists(src):
                shutil.copy(src, dst)
                print(f'  ✅ {fname}')
        restored = True
        print('Landmark data restored from Drive.\n')

if not os.path.exists(CROPS_DIR):
    drive_crops = DRIVE_CROPS_DIR
    if os.path.exists(drive_crops) and any(Path(drive_crops).iterdir()):
        print('Restoring image crops from Drive cache (~1-2 min)...')
        shutil.copytree(drive_crops, CROPS_DIR)
        print('✅ Crops restored from Drive.\n')

In [ ]:
if not restored and not os.path.exists(PROCESSED_NPY):
    # ── Need to download & preprocess from scratch ───────────────────────────
    if not os.path.exists(KAGGLE_JSON):
        raise FileNotFoundError(
            'kaggle.json not found at /content/kaggle.json.\n'
            'Steps:\n'
            '  1. Go to https://www.kaggle.com/settings → API → Create New Token\n'
            '  2. Upload kaggle.json via the Files panel (left sidebar)\n'
            '  3. Re-run this cell'
        )

    print('Installing kaggle CLI...')
    !pip install -q kaggle
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp {KAGGLE_JSON} {KAGGLE_DEST}
    !chmod 600 {KAGGLE_DEST}

    print('Downloading ASL Alphabet dataset (~1 GB)...')
    !kaggle datasets download -d grassknoted/asl-alphabet -p {DOWNLOAD_DIR} --unzip

    # Auto-detect extracted structure
    TRAIN_DIR = None
    best_count = 0
    for candidate in Path(DOWNLOAD_DIR).rglob('*'):
        if not candidate.is_dir():
            continue
        class_like = [d for d in candidate.iterdir()
                      if d.is_dir() and len(list(d.glob('*.jpg'))) > 0]
        if len(class_like) > best_count:
            best_count = len(class_like)
            TRAIN_DIR = str(candidate)
    if TRAIN_DIR is None:
        raise FileNotFoundError(f'Could not find class directories under {DOWNLOAD_DIR}.')
    print(f'✅ Detected training dir: {TRAIN_DIR} ({best_count} classes)')

    # Copy raw images
    os.makedirs(RAW_ASL_DIR, exist_ok=True)
    for class_dir in Path(TRAIN_DIR).iterdir():
        if class_dir.is_dir():
            dest = Path(RAW_ASL_DIR) / class_dir.name
            if not dest.exists():
                shutil.copytree(str(class_dir), str(dest))

    total_images = sum(len(list(d.glob('*.jpg'))) for d in Path(RAW_ASL_DIR).iterdir() if d.is_dir())
    print(f'✅ {total_images:,} images copied')

    # Run full preprocessing (landmarks + crops)
    print('\nRunning preprocessing pipeline (~10–20 min)...')
    !python {BACKEND_PATH}/src/preprocess.py --all --augment --aug_factor 3

    if not os.path.exists(PROCESSED_NPY):
        raise RuntimeError(f'Preprocessing failed — {PROCESSED_NPY} not found.')

    # Cache to Drive for future reconnects
    for fname in ['landmarks_all.npy', 'labels_all.npy', 'class_map.json']:
        src = f'{PROCESSED_DIR}/{fname}'
        dst = f'{DRIVE_DATA_DIR}/{fname}'
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
            print(f'  ✅ Cached {fname} to Drive')

    if os.path.exists(CROPS_DIR) and not any(Path(DRIVE_CROPS_DIR).iterdir()):
        print('Caching image crops to Drive (~1-2 min)...')
        shutil.copytree(CROPS_DIR, DRIVE_CROPS_DIR, dirs_exist_ok=True)
        print('✅ Crops cached to Drive')

# ── Load and verify ───────────────────────────────────────────────────────────
X = np.load(PROCESSED_NPY)
y = np.load(LABELS_NPY)
print(f'\n✅ Data ready:')
print(f'   X shape: {X.shape}  (samples × 63 landmarks)')
print(f'   y shape: {y.shape}  ({len(set(y.tolist()))} unique classes)')

In [ ]:
# ── Cell 5: Verify GPU ────────────────────────────────────────────────────────
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('✅ GPU memory growth enabled — ready to train.')
else:
    print('⚠️  WARNING: No GPU detected.')
    print('   MLP: ~1h | BiLSTM: ~3-4h | CNN: ~8-12h on CPU. Not recommended.')
    print('   Go to Runtime → Change runtime type → T4 GPU')

---
## 1 — MLP Training
**Input:** 63-dim normalized landmark vector  
**Architecture:** Dense(512) → BN → ReLU → Dropout → Dense(256) → ... → Softmax(29)  
**Expected time:** ~15 min

In [ ]:
# ── Cell 6: Train MLP ─────────────────────────────────────────────────────────
import os, sys, shutil
from pathlib import Path

BACKEND_PATH = '/content/backend'
MODEL_PY     = f'{BACKEND_PATH}/src/model.py'

# ── Patch model.py in-place for Keras 3 compatibility ────────────────────────
with open(MODEL_PY) as f:
    _src = f.read()

_changed = False

# Fix 1: replace AdamW with Adam in compile_model
if 'keras.optimizers.AdamW' in _src:
    _src = _src.replace('keras.optimizers.AdamW', 'keras.optimizers.Adam')
    _changed = True
    print('  🔧 Patched: AdamW → Adam')

# Fix 2: use keras.regularizers not tensorflow.keras regularizers
if 'from tensorflow.keras import layers, regularizers' in _src:
    _src = _src.replace(
        'from tensorflow.keras import layers, regularizers',
        'from tensorflow.keras import layers\nimport keras.regularizers as regularizers'
    )
    _changed = True
    print('  🔧 Patched: regularizers import')

# Fix 3: remove set_global_policy call
if 'set_global_policy("mixed_float16")' in _src:
    _src = _src.replace(
        'tf.keras.mixed_precision.set_global_policy("mixed_float16")',
        '# set_global_policy disabled — breaks Functional models in Keras 3'
    )
    _changed = True
    print('  🔧 Patched: set_global_policy removed')

if _changed:
    with open(MODEL_PY, 'w') as f:
        f.write(_src)
    print('✅ model.py patched')
else:
    print('✅ model.py already up to date')

# Clear pycache and module cache
for _d in [f'{BACKEND_PATH}/src/__pycache__', f'{BACKEND_PATH}/configs/__pycache__']:
    if os.path.exists(_d):
        shutil.rmtree(_d)
for _mod in list(sys.modules.keys()):
    if any(x in _mod for x in ['train', 'configs', 'model', 'evaluate', 'src']):
        del sys.modules[_mod]

# ── Now train ────────────────────────────────────────────────────────────────
from configs.training_config import MLPConfig
from src.train import train_mlp

cfg = MLPConfig()
cfg.save_dir        = Path(DRIVE_MODELS_DIR)
cfg.log_dir         = Path('/content/logs/mlp')
cfg.mixed_precision = False   # keep off — avoids Keras 3 mixed precision issues
os.makedirs(cfg.log_dir, exist_ok=True)

print('MLP config:')
print(f'  hidden_dims:     {cfg.hidden_dims}')
print(f'  dropout_rate:    {cfg.dropout_rate}')
print(f'  epochs:          {cfg.epochs}')
print(f'  batch_size:      {cfg.batch_size}')
print(f'  learning_rate:   {cfg.learning_rate}')
print(f'  mixed_precision: {cfg.mixed_precision}')
print()

model_mlp = train_mlp(cfg)
print('\n✅ MLP training complete!')

In [ ]:
# ── Cell 7: Evaluate MLP ──────────────────────────────────────────────────────
import sys
from pathlib import Path

for mod in list(sys.modules.keys()):
    if 'evaluate' in mod:
        del sys.modules[mod]

import src.evaluate as ev
ev.MODELS_DIR = Path(DRIVE_MODELS_DIR)

results_mlp = ev.evaluate('asl_mlp', split='test')
print(f"\nMLP — test accuracy: {results_mlp['accuracy']*100:.1f}%")
print(f"       top-5 accuracy: {results_mlp['top5']*100:.1f}%")

---
## 2 — BiLSTM Training
**Input:** 30×63 landmark sequences (synthesized by tiling + jitter from static frames)  
**Architecture:** BiLSTM(128) → Dropout → BiLSTM(64) → GlobalAvgPool → Softmax(29)  
**Expected time:** ~45 min

In [ ]:
# ── Cell 8: Train BiLSTM ──────────────────────────────────────────────────────
import os, sys
from pathlib import Path

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['train', 'configs', 'model', 'evaluate']):
        del sys.modules[mod]

from configs.training_config import LSTMConfig
from src.train import train_lstm

cfg = LSTMConfig()
cfg.save_dir        = Path(DRIVE_MODELS_DIR)
cfg.log_dir         = Path('/content/logs/lstm')
cfg.mixed_precision = False   # BiLSTM + mixed precision can be unstable
os.makedirs(cfg.log_dir, exist_ok=True)

print('BiLSTM config:')
print(f'  lstm_units:      {cfg.lstm_units}')
print(f'  sequence_len:    {cfg.sequence_len}')
print(f'  bidirectional:   {cfg.bidirectional}')
print(f'  dropout_rate:    {cfg.dropout_rate}')
print(f'  gradient_clip:   {cfg.gradient_clip}')
print(f'  epochs:          {cfg.epochs}')
print(f'  batch_size:      {cfg.batch_size}')
print(f'  learning_rate:   {cfg.learning_rate}')
print()
print('Note: Sequences synthesized from static frames (tile + jitter).')
print()

model_lstm = train_lstm(cfg)
print('\n✅ BiLSTM training complete!')

In [ ]:
# ── Cell 9: Evaluate BiLSTM ───────────────────────────────────────────────────
import sys
from pathlib import Path

for mod in list(sys.modules.keys()):
    if 'evaluate' in mod:
        del sys.modules[mod]

import src.evaluate as ev
ev.MODELS_DIR = Path(DRIVE_MODELS_DIR)

results_lstm = ev.evaluate('asl_lstm', split='test')
print(f"\nBiLSTM — test accuracy: {results_lstm['accuracy']*100:.1f}%")
print(f"          top-5 accuracy: {results_lstm['top5']*100:.1f}%")

---
## 3 — MobileNetV3 CNN Training (Two-Phase Fine-Tuning)
**Input:** 224×224×3 RGB hand crop images  
**Architecture:** MobileNetV3Small (frozen base) → Dropout → Dense(29, softmax)  
**Phase 1** (~20 epochs): train head only (frozen base)  
**Phase 2** (~40 epochs): unfreeze top layers from index 80, very low LR (1e-5)  
**Expected time:** ~2 h

In [ ]:
# ── Cell 10: Train MobileNetV3 CNN ───────────────────────────────────────────
import os, sys
from pathlib import Path

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['train', 'configs', 'model', 'evaluate']):
        del sys.modules[mod]

from configs.training_config import CNNConfig
from src.train import train_cnn

cfg = CNNConfig()
cfg.save_dir        = Path(DRIVE_MODELS_DIR)
cfg.log_dir         = Path('/content/logs/cnn')
cfg.mixed_precision = True   # Safe for CNN on T4
os.makedirs(cfg.log_dir, exist_ok=True)

print('CNN config:')
print(f'  base_model:      {cfg.base_model}')
print(f'  input_shape:     {cfg.input_shape}')
print(f'  fine_tune_from:  {cfg.fine_tune_from}')
print(f'  Phase 1:         {cfg.phase1_epochs} epochs  lr={cfg.phase1_lr}')
print(f'  Phase 2:         {cfg.phase2_epochs} epochs  lr={cfg.phase2_lr}')
print(f'  mixed_precision: {cfg.mixed_precision}')
print()

model_cnn = train_cnn(cfg)
if model_cnn is None:
    print('❌ CNN training failed — check that image crops exist in CROPS_DIR.')
else:
    print('\n✅ CNN training complete!')

In [ ]:
# ── Cell 11: Evaluate CNN ─────────────────────────────────────────────────────
import sys
from pathlib import Path

for mod in list(sys.modules.keys()):
    if 'evaluate' in mod:
        del sys.modules[mod]

import src.evaluate as ev
ev.MODELS_DIR = Path(DRIVE_MODELS_DIR)

results_cnn = ev.evaluate('asl_mobilenet', split='test')
print(f"\nCNN — test accuracy: {results_cnn['accuracy']*100:.1f}%")
print(f"       top-5 accuracy: {results_cnn['top5']*100:.1f}%")

---
## 4 — Results Summary

In [ ]:
# ── Cell 12: TensorBoard (all models) ────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir /content/logs

In [ ]:
# ── Cell 13: Final summary ────────────────────────────────────────────────────
import os, numpy as np, tensorflow as tf
from pathlib import Path

print('=' * 60)
print('  SignSense AI — Training Complete')
print('=' * 60)

# Results table
print(f'\n{"Model":<20} {"Test Acc":>10} {"Top-5":>8}')
print('-' * 40)
for name, r in [('MLP',           results_mlp),
                ('BiLSTM',        results_lstm),
                ('MobileNetV3',   results_cnn)]:
    print(f'{name:<20} {r["accuracy"]*100:>9.1f}%  {r["top5"]*100:>7.1f}%')

# Drive files
print(f'\nFiles saved to Drive ({DRIVE_MODELS_DIR}):')
for fname in sorted(os.listdir(DRIVE_MODELS_DIR)):
    size_mb = os.path.getsize(os.path.join(DRIVE_MODELS_DIR, fname)) / 1e6
    print(f'  {fname:<35} {size_mb:>6.1f} MB')

# Sanity check — load each and run a dummy prediction
print('\nSanity checks:')
checks = [
    ('asl_mlp.keras',       np.zeros((1, 63),          dtype=np.float32)),
    ('asl_lstm.keras',      np.zeros((1, 30, 63),      dtype=np.float32)),
    ('asl_mobilenet.keras', np.zeros((1, 224, 224, 3), dtype=np.float32)),
]
for fname, dummy in checks:
    fpath = os.path.join(DRIVE_MODELS_DIR, fname)
    if os.path.exists(fpath):
        m = tf.keras.models.load_model(fpath)
        pred = m.predict(dummy, verbose=0)
        status = '✅' if abs(pred.sum() - 1.0) < 0.01 else '❌'
        print(f'  {status} {fname:<35} output sum={pred.sum():.4f}')
    else:
        print(f'  ⚠️  {fname} not found on Drive')

print('\n✅ All done! Copy the Google Drive sharing links for each .keras file')
print('   and set GDRIVE_MODEL_URL_* in your backend .env')